# Interactive (live): watch a gravel river adjust in real time

Unlike the [equilibrium demo](interactive_single_segment.ipynb), this one **runs continuously**. While it's playing, drag the sliders and watch the long profile respond *transiently*: turn **sediment** up and it aggrades and steepens; drop **base level** and an incision wave climbs upstream. GRLP steps forward in your browser (Pyodide), frame by frame.

Run all cells, then press **▶ Play**, drag the sliders, and use **Reset** to restart from equilibrium.

In [ ]:
%pip install -q grlp networkx ipywidgets bqplot

In [ ]:
import numpy as np
import bqplot as bq
import ipywidgets as widgets

import grlp

YEAR = 31556926.   # seconds per year
DT = 10 * YEAR     # 10 years advanced per animation frame


def make_equilibrium(Qw, Qs, zbl):
    """A single segment started at steady state for the given inputs."""
    lp = grlp.LongProfile()
    lp.basic_constants()
    lp.bedload_lumped_constants()
    lp.set_hydrologic_constants()
    lp.set_x(dx=1000., nx=60, x0=1000.)
    lp.set_z(S0=-1e-2, z1=zbl)
    lp.set_Q(Qw)
    lp.set_B(100.)
    lp.set_niter(3)
    lp.set_uplift_rate(0.)
    lp.set_z_bl(zbl)
    lp.set_Qs_input_upstream(Qs)
    lp.evolve_threshold_width_river(nt=10, dt=1e13)
    return lp


Q0, QS0, ZBL0 = 100., 0.02, 0.
lp = make_equilibrium(Q0, QS0, ZBL0)
sim_t = 0.

# bqplot renders through the widget protocol, so each frame just reassigns the
# line's y-data and the existing plot updates in place — no per-frame image
# rendering, which keeps the animation smooth in the browser kernel (Pyodide).
x_km = lp.x / 1000.
x_sc = bq.LinearScale(min=float(x_km.min()), max=float(x_km.max()))
y_sc = bq.LinearScale(min=-120., max=1300.)
profile = bq.Lines(x=x_km, y=lp.z, scales={'x': x_sc, 'y': y_sc},
                   colors=['#1f77b4'])
base = bq.Lines(x=[float(x_km.min()), float(x_km.max())], y=[ZBL0, ZBL0],
                scales={'x': x_sc, 'y': y_sc},
                colors=['gray'], line_style='dashed')
ax_x = bq.Axis(scale=x_sc, label='Downstream distance [km]')
ax_y = bq.Axis(scale=y_sc, orientation='vertical', label='Elevation [m]')
fig = bq.Figure(marks=[profile, base], axes=[ax_x, ax_y], title='t = 0.0 kyr',
                layout=widgets.Layout(width='720px', height='420px'))

Qw  = widgets.FloatSlider(value=Q0,   min=20, max=600, step=20,
                          description='Water discharge $Q$ [m³/s]',
                          style={'description_width': '340px'},
                          layout=widgets.Layout(width='760px'))
Qs  = widgets.FloatSlider(value=QS0,  min=0.005, max=0.06, step=0.005,
                          description='Bed-load sediment input $Q_s$ [m³/s]',
                          readout_format='.3f',
                          style={'description_width': '340px'},
                          layout=widgets.Layout(width='760px'))
zbl = widgets.FloatSlider(value=ZBL0, min=-100, max=100, step=5,
                          description='Base level [m]',
                          style={'description_width': '340px'},
                          layout=widgets.Layout(width='760px'))

# `Play` emits a tick on a timer; each tick advances the model one step. This
# event-driven ticker replaces the background loop that a browser kernel cannot
# run.
play  = widgets.Play(value=0, min=0, max=1_000_000, step=1, interval=100)
reset = widgets.Button(description='Reset', icon='refresh')


def step(change):
    """Advance one frame, reading the sliders as live boundary conditions."""
    global sim_t
    lp.set_Q(Qw.value)
    lp.set_Qs_input_upstream(Qs.value)
    lp.set_z_bl(zbl.value)
    lp.evolve_threshold_width_river(nt=1, dt=DT)
    sim_t += DT
    profile.y = lp.z
    base.y = [zbl.value, zbl.value]
    fig.title = 't = %.1f kyr' % (sim_t / (1000. * YEAR))
play.observe(step, 'value')


def do_reset(_):
    """Restart from equilibrium for the current slider settings."""
    global lp, sim_t
    lp = make_equilibrium(Qw.value, Qs.value, zbl.value)
    sim_t = 0.
    profile.y = lp.z
    base.y = [zbl.value, zbl.value]
    fig.title = 't = 0.0 kyr'
reset.on_click(do_reset)

widgets.VBox([fig, widgets.HBox([play, reset]), Qw, Qs, zbl])
